In [ ]:
import time
import numpy as np
import pandas as pd
import requests

# 1. Diverse Indian Cities (Coastal, Industrial, Inland, Capital, Southern Tech Hubs)
# Ensures distinct natural clusters for Unit 6 (K-Means, Hierarchical, DBSCAN)
CITIES = {
    "Pune": {"lat": 18.5204, "lon": 73.8567, "zone": "West_Deccan"},
    "Mumbai": {"lat": 19.0760, "lon": 72.8777, "zone": "West_Coastal"},
    "Nagpur": {"lat": 21.1458, "lon": 79.0882, "zone": "Central_Inland"},
    "Delhi": {"lat": 28.6139, "lon": 77.2090, "zone": "North_IndoGangetic"},
    "Bengaluru": {"lat": 12.9716, "lon": 77.5946, "zone": "South_Plateau"},
    "Chennai": {"lat": 13.0827, "lon": 80.2707, "zone": "South_Coastal"},
}

# 2. Full 1-Year Window (Captures Winter, Summer, Monsoon, and Post-Monsoon)
# Ended 8 days ago to guarantee ERA5 historical reanalysis data is 100% finalized
START_DATE = "2025-11-10"
END_DATE = "2026-11-10"

AQ_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
WEATHER_URL = "https://archive-api.open-meteo.com/v1/archive"

AQ_VARS = [
    "pm10",
    "pm2_5",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone",
    "aerosol_optical_depth",
    "dust",
    "uv_index",
    "us_aqi",
]

WEATHER_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "shortwave_radiation",
]


def fetch_city_1yr(city_name: str, info: dict) -> pd.DataFrame:
  print(f"Fetching 1-year AQ & Weather data for {city_name}...")

  aq_params = {
      "latitude": info["lat"],
      "longitude": info["lon"],
      "start_date": START_DATE,
      "end_date": END_DATE,
      "hourly": ",".join(AQ_VARS),
      "timezone": "Asia/Kolkata",
  }
  aq_resp = requests.get(AQ_URL, params=aq_params, timeout=45)
  aq_resp.raise_for_status()
  df_aq = pd.DataFrame(aq_resp.json()["hourly"])

  wx_params = {
      "latitude": info["lat"],
      "longitude": info["lon"],
      "start_date": START_DATE,
      "end_date": END_DATE,
      "hourly": ",".join(WEATHER_VARS),
      "timezone": "Asia/Kolkata",
  }
  wx_resp = requests.get(WEATHER_URL, params=wx_params, timeout=45)
  wx_resp.raise_for_status()
  df_wx = pd.DataFrame(wx_resp.json()["hourly"])

  df_city = pd.merge(df_aq, df_wx, on="time", how="inner")
  df_city.insert(0, "city", city_name)
  df_city.insert(1, "region_zone", info["zone"])
  df_city.insert(2, "latitude", info["lat"])
  df_city.insert(3, "longitude", info["lon"])
  return df_city


def get_season(month: int) -> str:
  if month in [12, 1, 2]:
    return "Winter"
  elif month in [3, 4, 5]:
    return "Summer"
  elif month in [6, 7, 8, 9]:
    return "Monsoon"
  else:
    return "Post_Monsoon"


def categorize_aqi_3class(aqi: float) -> int:
  # 0: Good/Moderate (<=100), 1: Sensitive/Unhealthy (101-150), 2: Severe (>150)
  if aqi <= 100:
    return 0
  elif aqi <= 150:
    return 1
  return 2


def main():
  all_dfs = []
  for city, info in CITIES.items():
    try:
      df_c = fetch_city_1yr(city, info)
      all_dfs.append(df_c)
      time.sleep(2.0)  # Polite delay for 1-year API payload
    except Exception as e:
      print(f"Error fetching {city}: {e}")

  df = pd.concat(all_dfs, ignore_index=True)

  # Temporal & Seasonal Feature Engineering
  df["time"] = pd.to_datetime(df["time"])
  df["month"] = df["time"].dt.month
  df["hour"] = df["time"].dt.hour
  df["day_of_week"] = df["time"].dt.dayofweek
  df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
  df["season"] = df["month"].apply(get_season)

  df = df.dropna(subset=["us_aqi"]).reset_index(drop=True)


  df["high_health_risk"] = (df["us_aqi"] > 100).astype(int)

  # 3. Unit 4 & 5 (Multi-class Classification): 'health_risk_level' (0, 1, 2)
  df["health_risk_level"] = df["us_aqi"].apply(categorize_aqi_3class)

  # Save full 1-year dataset (~52,500 rows)
  full_file = "india_1yr_air_quality_full_2026.csv"
  df.to_csv(full_file, index=False)

  df_sample = (
      df.groupby("city", group_keys=False)
      .apply(lambda x: x.sample(n=min(len(x), 2000), random_state=42))
      .reset_index(drop=True)
  )
  sample_file = "india_1yr_air_quality_sample12k_2026.csv"
  df_sample.to_csv(sample_file, index=False)

  print("\n================ DATASET SUMMARY ================")
  print(f"Full 1-Year Dataset : {full_file} -> {df.shape}")
  print(
      f"Exam-Ready 12k File : {sample_file} -> {df_sample.shape} (Recommended"
      " for fast SVM/Clustering)"
  )
  print("\nTarget 1 (Regression) - us_aqi stats:")
  print(df["us_aqi"].describe().round(2))
  print("\nTarget 2 (Binary Classification) - high_health_risk:")
  print(df["high_health_risk"].value_counts())
  print("\nTarget 3 (Multi-Class) - health_risk_level:")
  print(df["health_risk_level"].value_counts())


if __name__ == "__main__":
  main()

Fetching 1-year AQ & Weather data for Pune...
Fetching 1-year AQ & Weather data for Mumbai...
Fetching 1-year AQ & Weather data for Nagpur...
Fetching 1-year AQ & Weather data for Delhi...
Fetching 1-year AQ & Weather data for Bengaluru...
Fetching 1-year AQ & Weather data for Chennai...


/tmp/ipykernel_524/3648767919.py:145: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 2000), random_state=42))



================ DATASET SUMMARY ================
Full 1-Year Dataset : india_1yr_air_quality_full_2026.csv -> (52704, 31)
Exam-Ready 12k File : india_1yr_air_quality_sample12k_2026.csv -> (12000, 31) (Recommended for fast SVM/Clustering)

Target 1 (Regression) - us_aqi stats:
count    52704.00
mean       113.75
std         80.84
min         18.00
25%         65.00
50%         94.00
75%        147.00
max        915.00
Name: us_aqi, dtype: float64

Target 2 (Binary Classification) - high_health_risk:
high_health_risk
0    28906
1    23798
Name: count, dtype: int64

Target 3 (Multi-Class) - health_risk_level:
health_risk_level
0    28906
2    12450
1    11348
Name: count, dtype: int64
